# 05 — Walk-Forward Testing and Robustness

**Objective:** test the strategy on unseen data using rolling training and test windows.

For each window I estimate the hedge ratio using past data only, then apply the strategy to the next test period.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm

from statsmodels.tsa.stattools import coint

STOCK_A = "KO"
STOCK_B = "PEP"

TRAIN_DAYS = 504
TEST_DAYS = 63
LOOKBACK = 60
ENTRY_Z = 2.0
EXIT_Z = 0.5
COST = 0.0005

PROJECT_ROOT = Path("..")
RESULTS_DIR = PROJECT_ROOT / "results" / "tables"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

prices = yf.download(
    [STOCK_A, STOCK_B],
    start="2018-01-01",
    end="2026-01-01",
    auto_adjust=True,
    progress=False
)["Close"].dropna()

## Backtest functions

In [ ]:
def estimate_hedge_ratio(train):
    model = sm.OLS(
        train[STOCK_A],
        sm.add_constant(train[STOCK_B])
    ).fit()

    return model.params["const"], model.params[STOCK_B]


def make_positions(zscore, entry, exit):
    positions = pd.Series(0.0, index=zscore.index)
    current = 0

    for i, z in enumerate(zscore):
        if np.isnan(z):
            continue

        if current == 0:
            if z < -entry:
                current = 1
            elif z > entry:
                current = -1
        elif abs(z) < exit:
            current = 0

        positions.iloc[i] = current

    return positions


def walk_forward(prices, lookback=60, entry=2.0, exit=0.5, cost=0.0005):
    all_test_returns = []
    windows = []

    start = 0

    while start + TRAIN_DAYS + TEST_DAYS <= len(prices):
        train_end = start + TRAIN_DAYS
        test_end = train_end + TEST_DAYS

        train = prices.iloc[start:train_end]
        test = prices.iloc[train_end:test_end]

        alpha, beta = estimate_hedge_ratio(train)
        _, coint_pvalue, _ = coint(train[STOCK_A], train[STOCK_B])

        history_start = max(0, train_end - lookback)
        combined = prices.iloc[history_start:test_end]

        spread = combined[STOCK_A] - (alpha + beta * combined[STOCK_B])
        zscore = (
            spread - spread.rolling(lookback).mean()
        ) / spread.rolling(lookback).std()

        position = make_positions(zscore, entry, exit).shift(1).fillna(0)

        returns_a = combined[STOCK_A].pct_change()
        returns_b = combined[STOCK_B].pct_change()

        pair_return = (
            returns_a - beta * returns_b
        ) / (1 + abs(beta))

        strategy_return = position * pair_return
        turnover = position.diff().abs().fillna(0)
        strategy_return -= turnover * cost

        test_returns = strategy_return.reindex(test.index).dropna()
        all_test_returns.append(test_returns)

        windows.append({
            "test_start": test.index[0],
            "test_end": test.index[-1],
            "beta": beta,
            "coint_pvalue": coint_pvalue,
            "test_return": (1 + test_returns).prod() - 1
        })

        start += TEST_DAYS

    return pd.concat(all_test_returns), pd.DataFrame(windows)


def metrics(returns):
    equity = (1 + returns).cumprod()
    total = equity.iloc[-1] - 1
    annualised = (1 + total) ** (252 / len(returns)) - 1
    volatility = returns.std() * np.sqrt(252)
    sharpe = annualised / volatility if volatility != 0 else np.nan
    drawdown = equity / equity.cummax() - 1

    return {
        "total_return": total,
        "annualised_return": annualised,
        "annualised_volatility": volatility,
        "sharpe_ratio": sharpe,
        "max_drawdown": drawdown.min()
    }

## Out-of-sample results

In [ ]:
final_oos_returns, window_results = walk_forward(
    prices,
    lookback=LOOKBACK,
    entry=ENTRY_Z,
    exit=EXIT_Z,
    cost=COST
)

result = metrics(final_oos_returns)

for key, value in result.items():
    print(f"{key}: {value:.4f}")

In [ ]:
equity = (1 + final_oos_returns).cumprod()

equity.plot(figsize=(10, 4))
plt.title("Walk-Forward Out-of-Sample Equity Curve")
plt.xlabel("Date")
plt.ylabel("Growth of 1 unit")
plt.grid(alpha=0.3)
plt.show()

## Parameter sensitivity

I vary the lookback and entry threshold to check whether the result depends on one specific parameter choice.

In [ ]:
rows = []

for lookback in [40, 60, 90]:
    for entry in [1.5, 2.0, 2.5]:
        r, _ = walk_forward(
            prices,
            lookback=lookback,
            entry=entry,
            exit=EXIT_Z,
            cost=COST
        )

        m = metrics(r)

        rows.append({
            "lookback": lookback,
            "entry_z": entry,
            "annualised_return": m["annualised_return"],
            "sharpe_ratio": m["sharpe_ratio"],
            "max_drawdown": m["max_drawdown"]
        })

sensitivity_results = pd.DataFrame(rows)
sensitivity_results

## Transaction-cost sensitivity

In [ ]:
rows = []

for cost in [0.0, 0.00025, 0.0005, 0.001]:
    r, _ = walk_forward(
        prices,
        lookback=LOOKBACK,
        entry=ENTRY_Z,
        exit=EXIT_Z,
        cost=cost
    )

    m = metrics(r)

    rows.append({
        "cost": cost,
        "annualised_return": m["annualised_return"],
        "sharpe_ratio": m["sharpe_ratio"],
        "max_drawdown": m["max_drawdown"]
    })

cost_results = pd.DataFrame(rows)
cost_results

## Save results for the final notebook

In [ ]:
final_oos_returns.to_csv(RESULTS_DIR / "oos_returns.csv")
window_results.to_csv(RESULTS_DIR / "window_results.csv", index=False)
sensitivity_results.to_csv(RESULTS_DIR / "sensitivity_results.csv", index=False)
cost_results.to_csv(RESULTS_DIR / "cost_results.csv", index=False)

print("Results saved to:", RESULTS_DIR)